<a href="https://colab.research.google.com/github/SanviBallal/1MS25IS107_assignment2/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam

In [3]:
import os
from google.colab import userdata

# Use the exact Secret name shown in your Colab Secrets panel.
# If your Secret is named GOOGLE_API_KEY:
api_key = userdata.get("GOOGLE_API_KEY")

# Remove accidental spaces or newlines
api_key = "".join(api_key.split())

# Make the key available to the Gemini SDK
os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully:", bool(api_key))

API key loaded successfully: True


In [4]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "fill_in1":(
        "Artificial intelligence enables computers to perform tasks that usually require human intelligence. "
        "These tasks include learning, reasoning, problem-solving, and understanding language. "
        "AI systems use algorithms and data to identify patterns and make decisions. "
        "Artificial intelligence is used in education, healthcare, transportation, and many other fields."
    ),
    "fill_in2":(
        "Machine learning is a field of artificial intelligence that allows computers to learn from data. "
        "A machine learning model identifies patterns in examples and uses them to make predictions. "
        "Supervised learning uses labeled data, while unsupervised learning works with unlabeled data. "
        "Reinforcement learning allows an agent to learn through rewards and penalties."
    ),
    "fill_in3":(
        "Generative artificial intelligence creates new content based on patterns learned from training data. "
        "It can generate text, images, audio, and computer code. "
        "Large language models are a type of generative AI that can understand and produce human-like text. "
        "Generative AI can support writing, summarization, brainstorming, and programming."
    ),
    "fill_in4":(
        "An AI agent is a system that observes its environment and takes actions to achieve a goal. "
        "A ReAct agent combines reasoning with actions to solve a task step by step. "
        "The agent can use tools such as search to gather useful information. "
        "The information retrieved by a tool helps the agent decide what action to take next."
    )
}

In [5]:
# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(
            1 for word in words
            if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []

    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Sanity check for document search
print(search_documents("artificial intelligence"))


[Source: fill_in3]
Generative artificial intelligence creates new content based on patterns learned from training data. It can generate text, images, audio, and computer code. Large language models are a type of generative AI that can understand and produce human-like text. Generative AI can support writing, summarization, brainstorming, and programming.

[Source: fill_in2]
Machine learning is a field of artificial intelligence that allows computers to learn from data. A machine learning model identifies patterns in examples and uses them to make predictions. Supervised learning uses labeled data, while unsupervised learning works with unlabeled data. Reinforcement learning allows an agent to learn through rewards and penalties.


In [6]:
# @title Ask Gemini to search for something

import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.8-flash",
    system_instruction="""
    You are a helpful AI research assistant.
    Use the search_documents tool to find relevant information
    before answering questions about the documents.
    """
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.8-flash


/tmp/ipykernel_2333/532025585.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:

import time
import os
from google import genai
from google.colab import userdata
from google.genai import types

# --------------------------------------------------
# 1. Load API Key from Google Colab Secrets
# --------------------------------------------------

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found. Check your Colab Secrets."
    )

os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully.")

# --------------------------------------------------
# 2. Create Gemini Client
# --------------------------------------------------

client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.5-flash-lite"

# --------------------------------------------------
# 3. Configure Search Tool
# --------------------------------------------------

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search documents to find the most relevant information.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        )
    ]
)

# --------------------------------------------------
# 4. Configure Gemini
# --------------------------------------------------

config = types.GenerateContentConfig(
    system_instruction=(
        "You are a document research assistant. "
        "For every question about the supplied documents, "
        "you MUST call the search_documents tool before answering. "
        "Do not answer from memory. Use the search results "
        "to prepare your final answer."
    ),
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)

# --------------------------------------------------
# 5. Retry Function
# --------------------------------------------------

def send_with_retry(contents, max_retries=5):

    for attempt in range(max_retries):

        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as e:

            err_str = str(e)

            if ("429" in err_str or "503" in err_str) and attempt < max_retries - 1:

                wait = 20 * (attempt + 1)

                print(
                    f"Temporary API issue. "
                    f"Waiting {wait} seconds before retrying..."
                )

                time.sleep(wait)

            else:
                raise

# --------------------------------------------------
# 6. Main ReAct Agent
# --------------------------------------------------

def run_agent(question: str, max_steps: int = 8) -> str:

    print(f"Question: {question}")
    print("-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):

        response = send_with_retry(contents)

        model_content = response.candidates[0].content

        # Preserve model response
        contents.append(model_content)

        # Identify tool calls
        calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        # Display final answer if no tool is called
        if not calls:

            print("\nFinal answer:")
            print(response.text)

            return response.text

        result_parts = []

        # Execute search tool
        for call in calls:

            args = dict(call.args)

            print(
                f"Step {step}: model calls "
                f"{call.name}({args})"
            )

            result = search_documents(**args)

            print(
                f"Search result preview: {str(result)[:150]}..."
            )

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # Send tool results back to Gemini
        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    return "Agent stopped: maximum steps reached."

# --------------------------------------------------
# 7. Test the ReAct Agent
# --------------------------------------------------

_ = run_agent(
    "Search the documents for information about machine learning. "
    "Use the search_documents tool and explain the results."
)


API key loaded successfully.
Question: Search the documents for information about machine learning. Use the search_documents tool and explain the results.
--------------------------------------------------
Step 1: model calls search_documents({'query': 'machine learning'})
Search result preview: [Source: fill_in2]
Machine learning is a field of artificial intelligence that allows computers to learn from data. A machine learning model identifie...

Final answer:
Based on the search results, here is the information about machine learning:

* **Definition:** Machine learning is a field of artificial intelligence that enables computers to learn from data. 
* **Function:** A machine learning model identifies patterns in examples and uses those patterns to make predictions.
* **Types of Machine Learning:**
  * **Supervised learning:** Uses labeled data.
  * **Unsupervised learning:** Works with unlabeled data.
  * **Reinforcement learning:** Allows an agent to learn through a system of rewar

In [8]:

# @title Ask follow up questions

questions = [
    # Question 1: Answered by the AI documents
    "What is artificial intelligence?",

    # Question 2: Answered by the AI documents
    "What are the different types of machine learning?",

    # Question 3: Answered by the AI documents
    "What is generative AI and what can it create?",

    # Question 4: Not answered by the database
    "How does quantum teleportation work?",

    # Question 5: General knowledge
    "Who invented the World Wide Web?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)

    print("\n" + "=" * 60)
    run_agent(q)



Question: What is artificial intelligence?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'artificial intelligence'})
Search result preview: [Source: fill_in3]
Generative artificial intelligence creates new content based on patterns learned from training data. It can generate text, images, ...
Step 2: model calls search_documents({'query': 'what is artificial intelligence definition'})
Search result preview: [Source: fill_in2]
Machine learning is a field of artificial intelligence that allows computers to learn from data. A machine learning model identifie...

Final answer:
Based on the provided documents, artificial intelligence (AI) enables computers to perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and understanding language. AI systems use algorithms and data to identify patterns and make decisions, and the technology is applied across various fields including education, heal

In [9]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)